# Laby 2

Use pytorch and create fully connected feed forward network using nn.module

Define the network as a class and train it with the explicit train loop on MNIST dataset

# Loading data and simple preprocessing

In [ ]:
import kagglehub
import pandas as pd
from keras.utils import to_categorical

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader


In [ ]:
path = kagglehub.dataset_download("zalando-research/fashionmnist")

df_train = pd.read_csv(f"{path}/fashion-mnist_train.csv")
df_test = pd.read_csv(f"{path}/fashion-mnist_test.csv")

print(df_train.shape)
print(df_test.shape)

Using Colab cache for faster access to the 'fashionmnist' dataset.
(60000, 785)
(10000, 785)


In [ ]:
X_train = df_train.copy()
y_train = X_train.pop("label")

X_test = df_test.copy()
y_test = X_test.pop("label")

In [ ]:
X_train.sample(2)

,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,pixel10,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
13108,0,0,0,0,0,2,0,0,1,1,...,20,2,0,0,0,0,0,0,0,0
13111,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,62,31,0,0


In [ ]:
y_train.sample(2)

,label
45364,8
19048,8


In [ ]:
X_train = X_train/255
X_test = X_test/255
X_test.sample(2)

,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,pixel10,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
5385,0.0,0.0,0.0,0.000000,0.0,0.003922,0.0,0.0,0.0,0.0,...,0.698039,0.619608,0.0,0.0,0.000000,0.007843,0.007843,0.011765,0.0,0.0
9495,0.0,0.0,0.0,0.003922,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.223529,0.321569,0.333333,0.090196,0.0,0.0


In [ ]:
#y_test = pd.DataFrame(to_categorical(y_test))
#y_train = pd.DataFrame(to_categorical(y_train))
y_test.sample(2)



,label
6957,0
2048,8


In [ ]:
# convertin to tensor
X_train = torch.tensor(X_train.values, dtype=torch.float32)
y_train = torch.tensor(y_train.values, dtype=torch.long)

X_test = torch.tensor(X_test.values, dtype = torch.float32)
y_test = torch.tensor(y_test.values, dtype = torch.long)

train_ds = TensorDataset(X_train, y_train)
test_ds = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=True)

# Definining neural network using Pytorch

In [ ]:
# defining the device
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


In [ ]:
class FCNet(nn.Module):
    def __init__(self, input_dim=784, hidden_dims=(256, 256), num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dims[0])
        self.fc2 = nn.Linear(hidden_dims[0], hidden_dims[1])
        self.fc3 = nn.Linear(hidden_dims[1], num_classes)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        return self.fc3(x)  # logits, bez softmax


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FCNet().to(device)
print(model)

FCNet(
  (fc1): Linear(in_features=784, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=256, bias=True)
  (fc3): Linear(in_features=256, out_features=10, bias=True)
  (dropout): Dropout(p=0.2, inplace=False)
)


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
epochs = 8

for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()

        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)
        loss.backward()

        optimizer.step()

        running_loss += loss.item() * X_batch.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(y_batch).sum().item()
        total += y_batch.size(0)

    avg_loss = running_loss / total
    acc = correct / total
    print(f"Epoch {epoch}: Loss={avg_loss:.4f}, Accuracy={acc*100:.2f}%")


Epoch 1: Loss=0.5458, Accuracy=80.40%
Epoch 2: Loss=0.3996, Accuracy=85.41%
Epoch 3: Loss=0.3667, Accuracy=86.50%
Epoch 4: Loss=0.3434, Accuracy=87.45%
Epoch 5: Loss=0.3259, Accuracy=87.95%
Epoch 6: Loss=0.3158, Accuracy=88.30%
Epoch 7: Loss=0.3053, Accuracy=88.66%
Epoch 8: Loss=0.2943, Accuracy=89.04%


In [ ]:
model.eval()

running_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
  for X_batch, y_batch in test_loader:
    X_batch, y_batch = X_batch.to(device), y_batch.to(device)

    outputs = model(X_batch)

    loss = criterion(outputs, y_batch)

    running_loss += loss.item() * X_batch.size(0)
    _, predicted = outputs.max(1)
    correct += predicted.eq(y_batch).sum().item()
    total += y_batch.size(0)

avg_loss = running_loss / total
acc = correct / total

print(f"Evalutaion:\n-Accuracy: {acc}\n-Loss:{avg_loss}")


Evalutaion:
-Accuracy: 0.8854
-Loss:0.3076826898097992


# Skip connections NN

In [ ]:
class SkipNN(nn.Module):
    def __init__(self, input_dim=784, num_of_layers=12, hidden_dims=256, num_classes=10):
      super().__init__()
      self.inputLayer = nn.Linear(input_dim, hidden_dims)

      self.layers = nn.ModuleList()
      for i in range(num_of_layers):
          self.layers.append(nn.Linear(hidden_dims, hidden_dims))


      self.outLayer = nn.Linear(hidden_dims, num_classes)

    def forward(self, x):
      ff_input = F.relu(self.inputLayer(x))

      ff_to_skip = [torch.zeros_like(ff_input), torch.zeros_like(ff_input)]
      ff_prev = ff_input

      for i ,layer in enumerate(self.layers):
        if i % 2 == 0:
          ff_x = layer(ff_prev) + ff_to_skip[0]
          ff_x = F.relu(ff_x)
          ff_to_skip[0] = ff_x
        else:
          ff_x = layer(ff_prev) + ff_to_skip[1]
          ff_x = F.relu(ff_x)
          ff_to_skip[1] = ff_x

        ff_prev = ff_x

      ff_out = F.relu(self.outLayer(ff_prev))

      logits = ff_out
      return logits

In [ ]:
model = SkipNN(num_of_layers=5)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

running_loss = 0.0
correct = 0
total = 0

for epoch in range(1, epochs + 1):
  for X_batch, y_batch in train_loader:
    model.train()

    optimizer.zero_grad()
    X_batch, y_batch = X_batch.to(device), y_batch.to(device)

    outputs = model(X_batch)

    loss = criterion(outputs, y_batch)
    loss.backward()

    optimizer.step()

    running_loss += loss.item() * X_batch.size(0)

    _, predicted = outputs.max(1)
    correct += predicted.eq(y_batch).sum().item()
    total += y_batch.size(0)

  avg_loss = running_loss / total
  acc = correct / total
  print(f"Epoch {epoch}: Loss={avg_loss:.4f}, Accuracy={acc*100:.2f}%")



Epoch 1: Loss=1.6935, Accuracy=39.52%
Epoch 2: Loss=1.6610, Accuracy=40.75%
Epoch 3: Loss=1.6435, Accuracy=41.36%
Epoch 4: Loss=1.6318, Accuracy=41.69%
Epoch 5: Loss=1.6226, Accuracy=41.94%
Epoch 6: Loss=1.6068, Accuracy=42.54%
Epoch 7: Loss=1.5760, Accuracy=43.77%
Epoch 8: Loss=1.5506, Accuracy=44.75%


In [ ]:
total_batches = 0
total_loss = 0
# model evalutation
for X_batch, y_batch in test_loader:
  model.eval()
  X_batch, y_batch = X_batch.to(device), y_batch.to(device)

  outputs = model(X_batch)

  loss = criterion(outputs, y_batch)

  total_batches += 1
  total_loss += loss.item()

  _, predicted = outputs.max(1)
  correct += predicted.eq(y_batch).sum().item()
  total += y_batch.size(0)

avg_loss = total_loss / total_batches
acc = correct / total

print(f"Evalutaion:\n-Accuracy: {acc}\n-Loss:{avg_loss}")


Evalutaion:
-Accuracy: 0.4489673469387755
-Loss:1.3812893735375373
